# AI Deep Interview Empath

Анализ глубинных-интервью

**Пайплайн:**

1. Загрузка аудио (приведение к 16 kHz моно).
2. **Диаризация** - кто и когда говорит (модель `pyannote/speaker-diarization-3.1`).
3. **Транскрипция** - что именно сказано (`faster-whisper large-v3`).
4. **Роли** - определяем интервьюера и гостя эвристикой на таймингах.
5. **Q/A-сегментация** - режем поток на пары «вопрос → ответ».
6. **Эмоции** (TODO) - SER-модель на сегментах гостя.
7. **Финальный результат** - Pydantic-контракт сохраняем в JSON.

## 0. Установка зависимостей и конфиг

#### **!!! Полный запуск всего ноутбука не сработает из-за кривых зависимостей pyannote.audio !!!**

В переменной `HF_TOKEN` нужно указать токен HuggingFace с принятым соглашением для модели pyannote — без него диаризационный pipeline не загрузится. Получить токен можно на huggingface.co/settings/tokens.



In [ ]:
%%capture

%pip uninstall -y numpy
%pip install -q --upgrade --no-cache-dir numpy numba librosa pyannote.audio
%cd /content
!git clone https://github.com/redm0unt/ai-podcast-empath.git

# import os
# os.kill(os.getpid(), 9)

In [ ]:
# !pip install torch torchaudio pydantic pyannote.audio faster-whisper

import json
import re
import uuid
from pathlib import Path
from typing import Optional

import torch
import torchaudio
from pydantic import BaseModel
from IPython.display import Audio, display


# === Конфиг ===
AUDIO_PATH = Path("tests/0.5min_emotional_sample_conversation.wav")
HF_TOKEN   = ""   # ← вставьте сюда HF-токен для pyannote
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Аудиофайл: {AUDIO_PATH}")
print(f"Устройство: {DEVICE}")

## 1. Загрузка аудио

Приводим аудио к **16 kHz моно** — это общий знаменатель для всех моделей в пайплайне: pyannote (диаризация), Whisper (ASR) и большинство SER-моделей. Файлы 44.1 kHz / 48 kHz / стерео нужно ресемплить и сводить в моно, иначе модели либо упадут, либо выдадут шум.

In [ ]:
def load_audio_16k_mono(path: Path) -> tuple[torch.Tensor, int]:
    waveform, sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:                    # стерео → моно усреднением каналов
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != 16000:                              # ресемплинг до 16 kHz
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
        sr = 16000
    return waveform, sr


waveform, sr = load_audio_16k_mono(AUDIO_PATH)
duration_sec = waveform.shape[-1] / sr
print(f"Длительность: {duration_sec:.1f} сек, sample rate: {sr} Hz, форма: {tuple(waveform.shape)}")

Audio(str(AUDIO_PATH))

## 2. Контракт данных

Описываем структуры, которые будут постепенно заполняться этапами пайплайна. Каждая модель — это маленький Pydantic-класс с типизированными полями. Зачем Pydantic, а не голые `dict`-ы:

- **типы видны сразу** — читая `Segment`, понятно что у него `start: float`, `text: Optional[str]`;
- **сериализация в JSON бесплатно** — `model_dump_json(indent=2)` отдаёт читаемый файл с правильными типами;
- **валидация на входе** — если в `Segment(start="abc")` подсунут строку, упадём здесь, а не через три ячейки.

**Главный контейнер — `InterviewAnalysis`** — собирает в себя всё:
- метаданные аудио (`AudioMeta`),
- список спикеров (`Speaker[]` с ролями),
- сегменты речи (`Segment[]` с текстом и словами),
- найденные вопросы (`Question[]` со ссылками на сегменты-ответы).

In [ ]:
class Speaker(BaseModel):
    id: str                         # "S0", "S1"
    label_src: Optional[str] = None # как звали в источнике, "SPEAKER_00"
    role: Optional[str] = None      # "interviewer" / "respondent"


class Word(BaseModel):
    '''Тайминги отдельного слова — нужны для резки сегментов на предложения.'''
    w: str
    start: float
    end: float
    conf: float


class EmotionAgg(BaseModel):
    '''Эмоция, агрегированная по сегменту.'''
    dominant: str
    scores: dict[str, float]


class Segment(BaseModel):
    '''Кусок речи одного спикера. Создаёт диаризация, остальные этапы дополняют.'''
    seg_id: str                            # "seg_0007"
    start: float
    end: float
    speaker: Optional[str] = None          # после диаризации
    text: Optional[str] = None             # после транскрипции
    words: Optional[list[Word]] = None     # после транскрипции
    emotion_agg: Optional[EmotionAgg] = None  # после SER


class SpeakerEmotion(BaseModel):
    '''Профиль эмоций одного спикера внутри ответа.'''
    speaker: str
    duration_sec: float
    emotion: EmotionAgg


class Question(BaseModel):
    q_id: str
    asked_by: str
    seg_ids: list[str]              # сегменты, из которых состоит сам вопрос
    start: float
    end: float
    text: str
    answer_seg_ids: list[str] = []  # сегменты гостя, идущие в ответ
    answer_emotions: list[SpeakerEmotion] = []


class AudioMeta(BaseModel):
    title: str
    uri: Optional[str] = None
    duration_sec: float
    sample_rate: int = 16000


class InterviewAnalysis(BaseModel):
    '''Главный контейнер — весь результат анализа одного интервью.'''
    job_id: str
    audio: AudioMeta
    speakers: list[Speaker] = []
    segments: list[Segment] = []
    questions: list[Question] = []


# Создаём пустой контракт — будем дописывать его на каждом этапе.
contract = InterviewAnalysis(
    job_id=str(uuid.uuid4()),
    audio=AudioMeta(
        title=AUDIO_PATH.stem,
        uri=str(AUDIO_PATH),
        duration_sec=duration_sec,
    ),
)

print(contract.model_dump_json(indent=2))

## 3. Диаризация — кто и когда говорит

**Speaker diarization** разделяет аудио на куски «один спикер — один кусок». Используем `pyannote/speaker-diarization-3.1` — нейросетевой пайплайн от pyannoteAI на основе кластеризации speaker-embeddings.

На выходе получаем список «turn»-ов вида `(start, end, label)`, где `label` — это абстрактный идентификатор спикера, который модель присвоила сама (`SPEAKER_00`, `SPEAKER_01`, ...). Мы переименуем их в `S0`, `S1` для читаемости.

**Один важный пост-процессинг — склейка коротких реплик.** Модель часто рвёт длинную монологичную реплику на 5-10 кусочков по 0.5–2 секунды (микропаузы между словами). Если оставить как есть, дальше анализировать «текст реплики» будет тяжело. Склеиваем подряд идущие turn'ы одного и того же спикера, если пауза между ними меньше `MERGE_GAP_SEC` секунд.

In [ ]:
from pyannote.audio import Pipeline

DIAR_MODEL    = "pyannote/speaker-diarization-3.1"
MERGE_GAP_SEC = 2.0  # склеиваем turn'ы одного спикера, разделённые паузой ≤ 2 сек


diar_pipeline = Pipeline.from_pretrained(DIAR_MODEL, token=HF_TOKEN or None)
if DEVICE == "cuda":
    diar_pipeline.to(torch.device("cuda"))

with torch.inference_mode():
    diarization = diar_pipeline({"waveform": waveform, "sample_rate": sr})

# Собираем все turn'ы и присваиваем человеческие ID (S0, S1, ...).
raw_turns: list[tuple[float, float, str]] = []
label_to_id: dict[str, str] = {}
for turn, _, label_src in diarization.speaker_diarization.itertracks(yield_label=True):
    if label_src not in label_to_id:
        label_to_id[label_src] = f"S{len(label_to_id)}"
    raw_turns.append((turn.start, turn.end, label_to_id[label_src]))
raw_turns.sort(key=lambda t: t[0])


def merge_same_speaker(turns, max_gap_sec):
    '''Склеиваем подряд идущие turn-ы одного спикера, если пауза <= max_gap_sec.'''
    if not turns:
        return []
    out = [turns[0]]
    for start, end, spk in turns[1:]:
        p_start, p_end, p_spk = out[-1]
        if spk == p_spk and start - p_end <= max_gap_sec:
            out[-1] = (p_start, max(p_end, end), p_spk)
        else:
            out.append((start, end, spk))
    return out


merged = merge_same_speaker(raw_turns, MERGE_GAP_SEC)

# Заполняем контракт.
contract.speakers = [
    Speaker(id=spk_id, label_src=label_src)
    for label_src, spk_id in label_to_id.items()
]
contract.segments = [
    Segment(seg_id=f"seg_{i:04d}", start=round(s, 3), end=round(e, 3), speaker=spk)
    for i, (s, e, spk) in enumerate(merged)
]

print(f"Найдено спикеров: {len(contract.speakers)}")
print(f"Сегментов после склейки: {len(contract.segments)} (до склейки: {len(raw_turns)})\n")
for seg in contract.segments[:10]:
    print(f"  [{seg.start:6.2f} — {seg.end:6.2f}]  {seg.speaker}")

## 4. Транскрипция — что именно сказано

Whisper переводит речь в текст и отдаёт пословные тайминги. Используем `faster-whisper large-v3` (это оптимизированная CTranslate2-реализация исходного Whisper-а — в 2-4 раза быстрее официального, без потери качества).

**Ключевое архитектурное решение:** прогоняем модель **один раз по всему файлу**, а не отдельно по каждому диар-сегменту. Почему так:

- Whisper плохо себя ведёт на коротких кусках <2 сек — может либо ничего не вернуть, либо галлюцинировать («Спасибо за просмотр!»).
- Прогон по всему файлу даёт правильную пунктуацию и контекст («А что вы думаете…» с предыдущим словом, а не в вакууме).
- VAD-фильтр Whisper'а сам выкидывает паузы.

После прогона получаем плоский список слов с таймингами. **Распределяем слова по диар-сегментам fusion-ом**: слово прикрепляется к сегменту, в чьи `[start, end]` попадает его центр. Реализация — два указателя за O(N+M), где N — слова, M — сегменты.

В итоге каждый `Segment` получает свой `text` и `words[]`, а его `start`/`end` подгоняются под границы реальных слов (если pyannote отдал сегмент с большой паузой по краям — обрежем).

In [ ]:
from faster_whisper import WhisperModel

ASR_MODEL_NAME = "large-v3"
LANGUAGE       = "ru"

compute_type = "float16" if DEVICE == "cuda" else "int8"
asr_model = WhisperModel(ASR_MODEL_NAME, device=DEVICE, compute_type=compute_type)

asr_segments, info = asr_model.transcribe(
    str(AUDIO_PATH),
    language=LANGUAGE,
    beam_size=5,
    word_timestamps=True,
    vad_filter=True,
)

# Собираем плоский список слов в хронологическом порядке.
all_words: list[Word] = []
for s in asr_segments:
    for w in (s.words or []):
        token = (w.word or "").strip()
        if not token:
            continue
        all_words.append(Word(
            w=token,
            start=round(float(w.start), 3),
            end=round(float(w.end), 3),
            conf=round(float(w.probability), 4),
        ))

print(f"Whisper выдал {len(all_words)} слов. Язык: {info.language} (p={info.language_probability:.2f})\n")


# Fusion двумя указателями: слово → сегмент по центру слова.
buckets: dict[str, list[Word]] = {s.seg_id: [] for s in contract.segments}
seg_idx = 0
n = len(contract.segments)

for word in all_words:
    center = (word.start + word.end) / 2.0
    while seg_idx < n and contract.segments[seg_idx].end < center:
        seg_idx += 1
    if seg_idx >= n:
        break
    seg = contract.segments[seg_idx]
    if seg.start <= center <= seg.end:
        buckets[seg.seg_id].append(word)
    # иначе слово в межсегментной паузе (музыка, шум) — отбрасываем


# Заполняем каждый сегмент его словами и текстом.
for seg in contract.segments:
    seg_words = buckets[seg.seg_id]
    if not seg_words:
        seg.text = ""
        seg.words = []
        continue
    seg.text  = " ".join(w.w for w in seg_words).strip()
    seg.start = seg_words[0].start
    seg.end   = seg_words[-1].end
    seg.words = seg_words


for seg in contract.segments[:5]:
    print(f"  {seg.speaker} [{seg.start:6.2f} — {seg.end:6.2f}]: {seg.text}")

## 5. Роли — интервьюер vs гость

Знаем, кто конкретно говорил (`S0`, `S1`), но не знаем, кто из них **ведёт**, а кто **отвечает**. На студийных подкастах с двумя спикерами это решается простой эвристикой без LLM — **голосование по трём признакам**:

1. **Меньше суммарного времени речи** — интервьюер задаёт короткие вопросы, гость даёт длинные ответы.
2. **Короче средняя реплика** — даже если интервьюер много говорит, его отдельные реплики короче.
3. **Начинает первым** — ведущий открывает интро.

Каждый признак даёт один голос «за это спикер S_i — интервьюер». У кого больше голосов — тот интервьюер. Tie-break: минимальное суммарное время.

На подкастах с >2 спикерами эвристика всё ещё работает (определяет ведущего), но для MVP проверяем только сценарий «интервью один-на-один».

In [ ]:
def pick_interviewer(speakers: list[Speaker], segments: list[Segment]) -> Optional[str]:
    '''3 признака голосуют за "это интервьюер", argmax голосов выигрывает.'''
    stats: dict[str, dict[str, float]] = {}
    for spk in speakers:
        durs = [s.end - s.start for s in segments if s.speaker == spk.id]
        if not durs:
            continue
        stats[spk.id] = {
            "total":       sum(durs),
            "avg":         sum(durs) / len(durs),
            "first_start": min(s.start for s in segments if s.speaker == spk.id),
        }
    if len(stats) < 2:
        return None

    votes = {sid: 0 for sid in stats}
    votes[min(stats, key=lambda s: stats[s]["total"])]       += 1
    votes[min(stats, key=lambda s: stats[s]["avg"])]         += 1
    votes[min(stats, key=lambda s: stats[s]["first_start"])] += 1

    max_votes = max(votes.values())
    leaders = [sid for sid, v in votes.items() if v == max_votes]
    if len(leaders) == 1:
        return leaders[0]
    return min(leaders, key=lambda s: stats[s]["total"])  # tie-break


interviewer_id = pick_interviewer(contract.speakers, contract.segments)

for spk in contract.speakers:
    spk.role = "interviewer" if spk.id == interviewer_id else "respondent"

for spk in contract.speakers:
    total = sum(s.end - s.start for s in contract.segments if s.speaker == spk.id)
    print(f"  {spk.id} → {spk.role:12s}  (всего речи: {total:.1f} сек)")

## 6. Сегментация на вопросы и ответы

Имеем: текст каждой реплики + словные тайминги + кто говорит + кто из них интервьюер. Осталось понять, **где в потоке вопросы и какие сегменты гостя на них отвечают**.

Подход:

1. **Режем реплики интервьюера на предложения** по знакам конца (`.`, `?`, `!`, `…`) — конец предложения определяем по последнему символу токена (Whisper приклеивает пунктуацию к слову).
2. **Определяем вопросительные предложения** по двум каналам:
   - явный `?` в конце (Whisper нередко его пропускает на русском, но не всегда);
   - первые 1-3 слова попадают в короткий whitelist вопросительных слов (`что`, `как`, `почему`, `расскажи`, ...) — это покрывает интонационные вопросы без `?`.
3. **Конечный автомат по сегментам:**
   - встретили вопросительное предложение интервьюера → открываем новый `Question`;
   - встретили сегмент гостя → приписываем его как `answer_seg_ids` к текущему открытому вопросу;
   - встретили второе вопросительное предложение **до того как пришёл хоть один сегмент ответа** → склеиваем в текущий вопрос (это серия из 2-3 уточнений одной репликой).

Эвристика без LLM — простая, объяснимая, и нормально работает на студийных интервью.

In [ ]:
QUESTION_HINTS = frozenset({
    "что", "как", "где", "когда", "почему", "зачем", "кто", "сколько",
    "какой", "какая", "какое", "какие",
    "расскажи", "расскажите", "опиши", "опишите", "поделись", "поделитесь",
})
SENT_END  = (".", "?", "!", "…")
TOKEN_RE  = re.compile(r"^[^\wа-яёА-ЯЁ]+|[^\wа-яёА-ЯЁ]+$")  # очистка пунктуации с краёв токена


def split_sentences(seg: Segment) -> list[dict]:
    '''Режем слова сегмента на предложения по знакам конца. Возвращает [{text, start, end}].'''
    if not seg.words:
        text = (seg.text or "").strip()
        return [{"text": text, "start": seg.start, "end": seg.end}] if text else []

    sentences, cur = [], []
    for w in seg.words:
        cur.append(w)
        token = (w.w or "").rstrip()
        if token and token[-1] in SENT_END:
            sentences.append({
                "text":  " ".join(x.w for x in cur).strip(),
                "start": cur[0].start,
                "end":   cur[-1].end,
            })
            cur = []
    if cur:
        sentences.append({
            "text":  " ".join(x.w for x in cur).strip(),
            "start": cur[0].start,
            "end":   cur[-1].end,
        })
    return sentences


def is_question(text: str) -> bool:
    '''Вопросительное предложение по двум каналам: "?" или вопросительное слово в первых 3 токенах.'''
    text = text.strip()
    if not text:
        return False
    if text.endswith("?"):
        return True
    for tok in text.split()[:3]:
        norm = TOKEN_RE.sub("", tok).lower()
        if norm and norm in QUESTION_HINTS:
            return True
    return False


# Конечный автомат по сегментам.
questions: list[Question] = []
current: Optional[Question] = None
counter = 0

for seg in contract.segments:
    if seg.speaker == interviewer_id:
        for sent in split_sentences(seg):
            if not is_question(sent["text"]):
                continue  # невопросительные комментарии интервьюера игнорим
            # Если у текущего вопроса ещё нет ответа гостя — склеиваем как продолжение.
            if current is not None and not current.answer_seg_ids:
                current.text = f"{current.text} {sent['text']}".strip()
                current.end  = sent["end"]
                if seg.seg_id not in current.seg_ids:
                    current.seg_ids.append(seg.seg_id)
                continue
            # Иначе — закрываем текущий, открываем новый.
            if current is not None:
                questions.append(current)
            current = Question(
                q_id=f"q_{counter:04d}",
                asked_by=interviewer_id,
                seg_ids=[seg.seg_id],
                start=sent["start"], end=sent["end"],
                text=sent["text"],
            )
            counter += 1
    else:  # сегмент гостя
        if current is None:
            continue  # гость говорит «в воздух» до первого вопроса — пропускаем
        if seg.seg_id not in current.answer_seg_ids:
            current.answer_seg_ids.append(seg.seg_id)

if current is not None:
    questions.append(current)

contract.questions = questions

print(f"Найдено вопросов: {len(contract.questions)}\n")
for q in contract.questions:
    print(f"  Q [{q.start:6.2f} — {q.end:6.2f}]: {q.text}")
    print(f"    → ответ: сегменты {q.answer_seg_ids}\n")

## 7. Эмоции в ответах гостя (TODO)

Этап намеренно оставлен как **заглушка** — реализация требует тяжёлой SER-модели (Speech Emotion Recognition), её скачивания (~1 GB) и аккуратной агрегации.

**Подход, который тут планируется:**

1. Грузим модель `wav2vec2-large-xlsr-53-russian-emotion` (или другую RU-SER из HuggingFace).
2. Для каждого сегмента гостя:
   - вырезаем waveform на интервале `[seg.start, seg.end]`,
   - прогоняем через модель,
   - сохраняем `EmotionAgg(dominant, scores={...})` в `Segment.emotion_agg`.
3. Для каждого `Question`: агрегируем эмоции по `answer_seg_ids`, **взвешивая по длительности сегмента** (более длинный сегмент важнее), записываем в `Question.answer_emotions`.

Ниже приведён код-скелет с закомментированной интеграцией — чтобы было видно, где какие хуки в контракте.

In [ ]:
# TODO: реальная реализация — раскомментируйте импорт модели и тело функции.

# from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
# SER_MODEL_NAME = "Aniemore/wav2vec2-xlsr-53-russian-emotion-recognition"
# ser_model     = AutoModelForAudioClassification.from_pretrained(SER_MODEL_NAME)
# ser_extractor = AutoFeatureExtractor.from_pretrained(SER_MODEL_NAME)
# LABELS = ser_model.config.id2label


def classify_segment_emotion(seg: Segment) -> EmotionAgg:
    '''Вырезает waveform сегмента и прогоняет через SER-модель.'''
    # samples = waveform[0, int(seg.start * sr) : int(seg.end * sr)]
    # inputs  = ser_extractor(samples.numpy(), sampling_rate=sr, return_tensors="pt")
    # with torch.inference_mode():
    #     logits = ser_model(**inputs).logits
    # probs  = logits.softmax(dim=-1)[0]
    # scores = {LABELS[i]: float(probs[i]) for i in range(len(LABELS))}
    # return EmotionAgg(dominant=max(scores, key=scores.get), scores=scores)
    raise NotImplementedError("Раскомментируй блок выше, установив transformers")


# for seg in contract.segments:
#     if seg.speaker != interviewer_id:           # только сегменты гостя
#         seg.emotion_agg = classify_segment_emotion(seg)
#
# # Агрегация по вопросам — взвешенная по длительности.
# for q in contract.questions:
#     by_speaker: dict[str, list[tuple[float, EmotionAgg]]] = {}
#     for sid in q.answer_seg_ids:
#         seg = next(s for s in contract.segments if s.seg_id == sid)
#         if seg.emotion_agg is None:
#             continue
#         by_speaker.setdefault(seg.speaker, []).append((seg.end - seg.start, seg.emotion_agg))
#     for spk, items in by_speaker.items():
#         total_dur = sum(d for d, _ in items)
#         agg_scores: dict[str, float] = {}
#         for d, em in items:
#             for label, score in em.scores.items():
#                 agg_scores[label] = agg_scores.get(label, 0.0) + score * d / total_dur
#         q.answer_emotions.append(SpeakerEmotion(
#             speaker=spk, duration_sec=total_dur,
#             emotion=EmotionAgg(dominant=max(agg_scores, key=agg_scores.get), scores=agg_scores),
#         ))

print("Этап эмоций оставлен как заглушка — раскомментируйте код выше, чтобы запустить SER.")

## 8. Финальный результат

Контракт `InterviewAnalysis` полностью заполнен. Сохраняем в JSON (`exclude_none=True` чтобы не засорять файл пустыми полями) и печатаем «человеческий» вид — список вопросов с их ответами.

In [ ]:
OUTPUT_PATH = Path(f"{AUDIO_PATH.stem}.analysis.json")
OUTPUT_PATH.write_text(
    contract.model_dump_json(indent=2, exclude_none=True),
    encoding="utf-8",
)
print(f"✅ Результат сохранён: {OUTPUT_PATH.resolve()}\n")


# Красивый вывод диалога.
seg_by_id = {s.seg_id: s for s in contract.segments}

print("=" * 70)
print(f"  {contract.audio.title}  ({contract.audio.duration_sec:.1f} сек)")
print("=" * 70)

for q in contract.questions:
    print(f"\n❓ ВОПРОС [{q.start:.1f}—{q.end:.1f} сек, q_id={q.q_id}]")
    print(f"   {q.text}")
    if q.answer_seg_ids:
        print(f"\n💬 ОТВЕТ ({len(q.answer_seg_ids)} сегментов):")
        for sid in q.answer_seg_ids:
            s = seg_by_id[sid]
            print(f"   [{s.start:6.2f}] {s.text}")
    else:
        print("   (без ответа)")